In [1]:
import numpy as np
import librosa
import random
import sounddevice as sd
import soundfile as sf
from skimage.transform import resize
from gammatone.gtgram import gtgram
from tensorflow.keras.models import load_model
import tensorflow as tf
from tensorflow.keras import backend as K


In [2]:
def extract_features(path, max_len=250):
    y, sr = librosa.load(path, sr=16000)

    # -------------------------
    # 1️⃣ MFCC Features (40)
    # -------------------------
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    if mfcc.shape[1] < max_len:
        mfcc = np.pad(mfcc, ((0,0),(0,max_len - mfcc.shape[1])), mode="constant")
    else:
        mfcc = mfcc[:, :max_len]

    # -------------------------
    # 2️⃣ Gammatone Features (64)
    # -------------------------
    gt = gtgram(
        y,
        fs=sr,
        window_time=0.025,
        hop_time=0.010,
        channels=64,
        f_min=50
    )

    gt = resize(gt, (64, max_len), mode="constant")

    combined = np.vstack([mfcc, gt])
    return np.expand_dims(combined, axis=-1)


In [3]:
from tensorflow.keras.models import load_model
import tensorflow as tf
from tensorflow.keras import backend as K

def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        pt = tf.where(tf.equal(y_true,1), y_pred, 1-y_pred)
        return -K.mean(alpha * K.pow(1-pt,gamma) * K.log(pt))
    return loss

model = load_model("covid_cough_model.h5", custom_objects={"loss": focal_loss()})
print("Model loaded successfully!")



Model loaded successfully!


In [4]:
import sounddevice as sd
import numpy as np
import soundfile as sf

FS = 16000
DURATION = 3   # seconds

def real_time_predict():
    print("🎙️ Speak / cough now...")

    audio = sd.rec(
        int(FS * DURATION),
        samplerate=FS,
        channels=1,
        dtype='float32'
    )
    sd.wait()

    audio = audio.flatten()
    sf.write("temp.wav", audio, FS)

    feat = extract_features("temp.wav")
    feat = np.expand_dims(feat, axis=0)

    pred = model.predict(feat, verbose=0)

    confidence = np.max(pred) * 100

    # ⚠️ Your requested logic
    if confidence < 97.50:
        result = "COVID Positive"
    else:
        result = "COVID Negative"

    print(f"🧠 Prediction: {result}")
    print(f"📊 Confidence: {confidence:.2f}%")


In [5]:
real_time_predict()


🎙️ Speak / cough now...
🧠 Prediction: COVID Negative
📊 Confidence: 98.72%
